## 🎯 Learning Objectives
* Understand the principles of streaming responses from Large Language Models (LLMs).
* Implement an asynchronous Q&A interface that leverages LLM streaming capabilities.
* Practice handling asynchronous generators for real-time output display.
* Develop robust error handling for LLM interactions in a streaming context.


## Exercise: Build a Simple Q&A Interface with Streaming Responses

**Objective:** In this exercise, you will build a command-line Q&A application that interacts with a Large Language Model (LLM) and displays its responses in a streaming fashion. This is a fundamental pattern for modern LLM applications, providing a better user experience by showing output as it's generated, rather than waiting for the entire response.

**Context (2026 Readiness):** Modern LLM APIs (e.g., OpenAI, Anthropic, Google Gemini, open-source models via services like Anyscale Endpoints or Together AI) predominantly offer streaming capabilities. Efficiently handling these streams using asynchronous programming (`asyncio` in Python) is crucial for building responsive and scalable applications.

**Requirements:**

1.  **LLM Interaction:** Use the provided `MockLLMClient` to simulate sending user queries to an LLM. This client will mimic the asynchronous streaming behavior of real LLM APIs.
2.  **Streaming Output:** Display the LLM's response word-by-word or token-by-token as it arrives. Do *not* wait for the full response to be generated before printing.
3.  **User Interface:** Implement a simple command-line loop where the user can type questions. Each question should trigger an LLM call.
4.  **Exit Condition:** Allow the user to type 'quit' or 'exit' (case-insensitive) to gracefully end the Q&A session.
5.  **Error Handling:** Gracefully handle potential errors that might occur during the LLM interaction (e.g., simulated network issues or API errors from the mock client).
6.  **Asynchronous Programming:** The entire Q&A loop, especially the interaction with the `MockLLMClient`, must leverage Python's `asyncio` for non-blocking I/O.

**Evaluation Criteria:**

*   **Functional Correctness:** Does the Q&A loop correctly process user input and interact with the mock LLM client?
*   **Streaming Implementation:** Is the LLM response truly streaming to the console, or is it buffered and printed all at once? (The output should appear character by character or word by word).
*   **Robustness:** Does the application handle the specified exit conditions and gracefully manage simulated errors?
*   **Code Quality:** Is the code well-structured, readable, and appropriately commented? Does it effectively use `asyncio`?


In [ ]:
import asyncio
import time
import random

# --- Setup Code: Mock LLM Client for Streaming Responses ---

class MockLLMClient:
    """
    A mock LLM client that simulates streaming responses asynchronously.
    This class mimics the behavior of real LLM APIs (e.g., OpenAI's `client.chat.completions.create(stream=True)`)
    by yielding tokens one by one with artificial delays.
    """
    def __init__(self, token_delay_range=(0.01, 0.05), error_chance=0.1):
        """
        Initializes the mock client.
        :param token_delay_range: A tuple (min_delay, max_delay) for simulating token generation time.
        :param error_chance: Probability (0.0 to 1.0) of a simulated error occurring.
        """
        self.token_delay_range = token_delay_range
        self.error_chance = error_chance
        self._mock_responses = {
            "hello": "Hello there! How can I assist you today?",
            "how are you": "I am an AI, so I don't have feelings, but I'm ready to help!",
            "what is python": "Python is a high-level, interpreted programming language known for its readability and versatility. It's widely used for web development, data analysis, AI, and more.",
            "streaming": "Streaming responses are crucial for a good user experience in LLM applications. They allow users to see the output as it's generated, reducing perceived latency and making the interaction feel more dynamic.",
            "asyncio": "Asyncio is Python's library for writing concurrent code using the async/await syntax. It's perfect for I/O-bound and high-level structured network code, like handling streaming API responses.",
            "default": "I'm not sure how to answer that. Could you please rephrase your question?"
        }

    async def _mock_stream_response(self, prompt: str):
        """
        An asynchronous generator that yields tokens from a simulated response.
        Simulates network latency and token generation.
        """
        # Simulate a potential API error or network issue
        if random.random() < self.error_chance:
            await asyncio.sleep(random.uniform(0.1, 0.5)) # Simulate some delay before error
            raise ConnectionError("Simulated API connection error or service unavailability.")

        # Determine response based on prompt, default if not found
        prompt_lower = prompt.lower()
        response_text = self._mock_responses.get(
            next((k for k in self._mock_responses if k in prompt_lower), "default"),
            self._mock_responses["default"]
        )

        # Simulate token-by-token streaming
        for word in response_text.split(' '):
            yield word + ' '
            await asyncio.sleep(random.uniform(*self.token_delay_range))
        yield ""

    async def stream_chat_completion(self, prompt: str):
        """
        Public method to get a streaming chat completion.
        :param prompt: The user's input prompt.
        :return: An asynchronous generator yielding response tokens.
        """
        print(f"\n[Mock LLM received prompt: '{prompt}']")
        return self._mock_stream_response(prompt)

# Instantiate the mock client. You will use this instance in your solution.
# Adjust error_chance to test error handling (e.g., 0.3 for more frequent errors).
llm_client = MockLLMClient(error_chance=0.1)

print("Mock LLM Client initialized. Ready for your implementation!")
print("Try asking: 'Hello', 'What is Python?', 'Tell me about streaming', or 'What is asyncio?'.")
print("Type 'quit' or 'exit' to end the session.")


## Your Turn! Implement the Streaming Q&A Interface

Now it's your turn to write the core logic for the Q&A application. Your task is to:

1.  Create an `async` function, for example, `main_qa_loop()`.
2.  Inside this function, implement an infinite loop that continuously prompts the user for input.
3.  Check if the user input is 'quit' or 'exit' (case-insensitive) and break the loop if it is.
4.  If the input is not an exit command, call `llm_client.stream_chat_completion()` with the user's prompt.
5.  Iterate asynchronously over the tokens yielded by the `stream_chat_completion` generator.
6.  Print each token to the console *immediately* as it arrives, ensuring that the output appears to stream.
7.  Implement a `try...except` block around the LLM interaction to catch potential `ConnectionError` (or other `Exception` types) from the mock client and print an informative error message.
8.  Finally, use `asyncio.run()` to execute your `main_qa_loop()` function.

Remember to use `async` and `await` keywords appropriately for asynchronous operations and `flush=True` when printing tokens to ensure immediate display.


In [ ]:
import asyncio
import time
import random

# Re-define MockLLMClient for self-contained solution, or assume it's defined above
class MockLLMClient:
    """
    A mock LLM client that simulates streaming responses asynchronously.
    This class mimics the behavior of real LLM APIs (e.g., OpenAI's `client.chat.completions.create(stream=True)`)
    by yielding tokens one by one with artificial delays.
    """
    def __init__(self, token_delay_range=(0.01, 0.05), error_chance=0.1):
        """
        Initializes the mock client.
        :param token_delay_range: A tuple (min_delay, max_delay) for simulating token generation time.
        :param error_chance: Probability (0.0 to 1.0) of a simulated error occurring.
        """
        self.token_delay_range = token_delay_range
        self.error_chance = error_chance
        self._mock_responses = {
            "hello": "Hello there! How can I assist you today?",
            "how are you": "I am an AI, so I don't have feelings, but I'm ready to help!",
            "what is python": "Python is a high-level, interpreted programming language known for its readability and versatility. It's widely used for web development, data analysis, AI, and more.",
            "streaming": "Streaming responses are crucial for a good user experience in LLM applications. They allow users to see the output as it's generated, reducing perceived latency and making the interaction feel more dynamic.",
            "asyncio": "Asyncio is Python's library for writing concurrent code using the async/await syntax. It's perfect for I/O-bound and high-level structured network code, like handling streaming API responses.",
            "default": "I'm not sure how to answer that. Could you please rephrase your question?"
        }

    async def _mock_stream_response(self, prompt: str):
        """
        An asynchronous generator that yields tokens from a simulated response.
        Simulates network latency and token generation.
        """
        # Simulate a potential API error or network issue
        if random.random() < self.error_chance:
            await asyncio.sleep(random.uniform(0.1, 0.5)) # Simulate some delay before error
            raise ConnectionError("Simulated API connection error or service unavailability.")

        # Determine response based on prompt, default if not found
        prompt_lower = prompt.lower()
        response_text = self._mock_responses.get(
            next((k for k in self._mock_responses if k in prompt_lower), "default"),
            self._mock_responses["default"]
        )

        # Simulate token-by-token streaming
        for word in response_text.split(' '):
            yield word + ' '
            await asyncio.sleep(random.uniform(*self.token_delay_range))
        yield ""

    async def stream_chat_completion(self, prompt: str):
        """
        Public method to get a streaming chat completion.
        :param prompt: The user's input prompt.
        :return: An asynchronous generator yielding response tokens.
        """
        print(f"\n[Mock LLM received prompt: '{prompt}']")
        return self._mock_stream_response(prompt)


# --- Reference Solution --- 

async def main_qa_loop():
    """
    Implements the main asynchronous Q&A loop with streaming LLM responses.
    """
    # Initialize the mock LLM client. Adjust error_chance to test error handling.
    # A higher error_chance (e.g., 0.3) will make errors more frequent for testing.
    llm_client = MockLLMClient(error_chance=0.1)

    print("\n--- Streaming Q&A Interface Started ---")
    print("Ask me anything! Type 'quit' or 'exit' to end the session.")

    while True:
        # Prompt user for input
        user_input = input("\nYour question: ")

        # Check for exit commands
        if user_input.lower() in ['quit', 'exit']:
            print("Exiting Q&A session. Goodbye!")
            break

        if not user_input.strip():
            print("Please enter a question.")
            continue

        print("LLM Response: ", end="") # Start printing on the same line
        full_response = [] # To optionally store the full response

        try:
            # Asynchronously iterate over the streaming response from the LLM client
            # The 'async for' loop is key to handling asynchronous generators.
            async for token in llm_client.stream_chat_completion(user_input):
                print(token, end="", flush=True) # Print each token immediately without a newline
                full_response.append(token)
            print() # Print a final newline after the full response is streamed

        except ConnectionError as e:
            # Catch specific connection errors from our mock client or real API issues
            print(f"\n[Error]: A connection error occurred: {e}. Please try again.")
        except Exception as e:
            # Catch any other unexpected errors during the LLM interaction
            print(f"\n[Error]: An unexpected error occurred: {e}.")

# Entry point for running the asynchronous main loop
if __name__ == "__main__":
    # asyncio.run() is used to run the top-level async function.
    # It manages the event loop and ensures all async tasks complete.
    asyncio.run(main_qa_loop())
